# TSP Baseline -- Concorde (Exact Solver)

Concorde is the gold-standard exact solver for TSP. It finds provably optimal solutions via branch-and-cut. Use it as ground truth for TSP-10 through TSP-100. For larger sizes it may be slow or infeasible within time limits.

## 1. Install Concorde
Install the `pyconcorde` wrapper.

In [ ]:
!pip install -q pyconcorde

## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted.")
except ImportError:
    print("[WARNING] Not running in Google Colab. Skipping Drive mount.")

## 3. Configuration

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

TSP_SIZES = list(range(10, 201, 10))
NUM_EVAL_INSTANCES = 128
TIME_LIMIT_SEC = 300
DISTANCE_SCALE = 100_000
DRIVE_DATA_DIR = '/content/drive/MyDrive/TSP_Data'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/TSP_Data/baseline_results'
SEED = 42

np.random.seed(SEED)

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"[OK] Results directory created/verified: {DRIVE_RESULTS_DIR}")

## 4. Solver Functions

In [ ]:
from concorde.tsp import TSPSolver
import signal

def compute_tour_length_euclidean(coords, tour):
    """
    Compute the Euclidean length of a TSP tour.

    Parameters
    ----------
    coords : np.ndarray
        Array of shape (N, 2) containing 2D coordinates.
    tour : list or np.ndarray
        Sequence of indices representing the tour.

    Returns
    -------
    float
        Total Euclidean distance of the tour.
    """
    if not tour:
        return float('inf')
    tour_coords = coords[tour]
    tour_coords = np.vstack([tour_coords, tour_coords[0]])
    diffs = np.diff(tour_coords, axis=0)
    distances = np.linalg.norm(diffs, axis=1)
    return float(np.sum(distances))

def solve_tsp_concorde(coords, time_limit_sec=TIME_LIMIT_SEC):
    """
    Solve a single TSP instance using Concorde exact solver.

    Parameters
    ----------
    coords : np.ndarray
        Array of shape (N, 2) containing 2D coordinates in [0, 1].
    time_limit_sec : float, optional
        Time limit in seconds (default is TIME_LIMIT_SEC).

    Returns
    -------
    dict
        Dictionary with keys: tour, tour_length, status, wall_time, is_optimal
    """
    n = len(coords)
    xs = (coords[:, 0] * DISTANCE_SCALE).astype(int)
    ys = (coords[:, 1] * DISTANCE_SCALE).astype(int)
    
    t0 = time.time()
    try:
        solver = TSPSolver.from_data(xs, ys, norm='EUC_2D')
        solution = solver.solve(time_bound=float(time_limit_sec), verbose=False)
        wall_time = time.time() - t0
        tour = list(solution.tour)
        tour_length = compute_tour_length_euclidean(coords, tour)
        is_optimal = solution.found_tour
        status = 'OPTIMAL' if is_optimal else 'FEASIBLE'
    except Exception as e:
        wall_time = time.time() - t0
        tour = []
        tour_length = float('inf')
        is_optimal = False
        status = f'ERROR: {str(e)[:50]}'
    
    return {
        'tour': tour,
        'tour_length': tour_length,
        'status': status,
        'wall_time': wall_time,
        'is_optimal': is_optimal,
    }

## 5. Quick Smoke Test

In [ ]:
print(f"-----------------------------------\n--- Quick Smoke Test (TSP-10) ---\n-----------------------------------")
dummy_coords = np.random.rand(10, 2)
res = solve_tsp_concorde(dummy_coords)
print(f"Tour: {res['tour']}")
print(f"Tour Length: {res['tour_length']:.4f}")
print(f"Status: {res['status']}")
print(f"Wall Time: {res['wall_time']:.4f} sec")
print(f"Is Optimal: {res['is_optimal']}")

## 6. Evaluate All TSP Sizes

In [ ]:
results = []
per_instance_records = []
tours_by_size = {}

for n in TSP_SIZES:
    dataset_path = os.path.join(DRIVE_DATA_DIR, f'tsp{n}_val.npy')
    
    if not os.path.exists(dataset_path):
        print(f"[WARNING] Dataset not found: {dataset_path}. Skipping TSP-{n}.")
        continue
        
    print(f"\n-----------------------------------\n--- Evaluating TSP-{n} ---\n-----------------------------------")
    dataset = np.load(dataset_path)
    if NUM_EVAL_INSTANCES is not None and len(dataset) > NUM_EVAL_INSTANCES:
        idx = np.random.choice(len(dataset), NUM_EVAL_INSTANCES, replace=False)
        dataset = dataset[idx]
        
    num_inst = len(dataset)
    print(f"Loaded {num_inst} instances.")
    
    tours = []
    tour_lengths = []
    wall_times = []
    statuses = []
    optimals = []
    
    for i, coords in enumerate(tqdm(dataset, desc=f"TSP-{n}")):
        res = solve_tsp_concorde(coords)
        tours.append(res['tour'])
        tour_lengths.append(res['tour_length'])
        wall_times.append(res['wall_time'])
        statuses.append(res['status'])
        optimals.append(res['is_optimal'])
        
        per_instance_records.append({
            'size': n,
            'instance_id': i,
            'tour_length': res['tour_length'],
            'wall_time': res['wall_time'],
            'status': res['status'],
            'is_optimal': res['is_optimal']
        })
        
    tours_by_size[n] = tours
    
    valid_lengths = [tl for tl in tour_lengths if tl != float('inf')]
    
    summary = {
        'TSP Size': n,
        'Instances': num_inst,
        'Tour Length (Mean)': np.mean(valid_lengths) if valid_lengths else float('nan'),
        'Tour Length (Std)': np.std(valid_lengths) if valid_lengths else float('nan'),
        'Tour Length (Min)': np.min(valid_lengths) if valid_lengths else float('nan'),
        'Tour Length (Max)': np.max(valid_lengths) if valid_lengths else float('nan'),
        'Time/inst (sec)': np.mean(wall_times),
        'Total Time (sec)': np.sum(wall_times),
        'Optimal %': 100 * np.mean(optimals),
    }
    results.append(summary)
    print(f"[OK] Finished TSP-{n}. Mean Length: {summary['Tour Length (Mean)']:.4f}, Mean Time: {summary['Time/inst (sec)']:.4f}s")

## 7. Summary Table

In [ ]:
if results:
    df_summary = pd.DataFrame(results)
    display(df_summary)
else:
    print(f"[WARNING] No results to summarize.")

## 8. Plots

In [ ]:
if results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sizes = df_summary['TSP Size']
    
    axes[0].plot(sizes, df_summary['Tour Length (Mean)'], marker='o', color='blue')
    axes[0].set_title('Mean Tour Length vs TSP Size')
    axes[0].set_xlabel('TSP Size (Nodes)')
    axes[0].set_ylabel('Mean Euclidean Tour Length')
    axes[0].grid(True, linestyle='--')
    
    axes[1].plot(sizes, df_summary['Time/inst (sec)'], marker='s', color='red')
    axes[1].set_title('Mean Time per Instance vs TSP Size')
    axes[1].set_xlabel('TSP Size (Nodes)')
    axes[1].set_ylabel('Time (sec)')
    axes[1].grid(True, linestyle='--')
    
    axes[2].plot(sizes, df_summary['Time/inst (sec)'], marker='^', color='green')
    axes[2].set_title('Mean Time per Instance (Log Scale)')
    axes[2].set_xlabel('TSP Size (Nodes)')
    axes[2].set_ylabel('Time (sec)')
    axes[2].set_yscale('log')
    axes[2].grid(True, linestyle='--')
    
    plt.tight_layout()
    plt.show()

## 9. Tour Length Distribution

In [ ]:
if per_instance_records:
    df_instances = pd.DataFrame(per_instance_records)
    df_valid = df_instances[df_instances['tour_length'] != float('inf')]
    
    plt.figure(figsize=(10, 6))
    df_valid.boxplot(column='tour_length', by='size', grid=True)
    plt.title('Tour Length Distribution per TSP Size')
    plt.suptitle('')
    plt.xlabel('TSP Size')
    plt.ylabel('Tour Length')
    plt.show()

## 10. Save Results

In [ ]:
if results:
    summary_csv_path = os.path.join(DRIVE_RESULTS_DIR, 'concorde_summary.csv')
    df_summary.to_csv(summary_csv_path, index=False)
    print(f"[OK] Saved summary to {summary_csv_path}")
    
    instance_csv_path = os.path.join(DRIVE_RESULTS_DIR, 'concorde_per_instance.csv')
    df_instances.to_csv(instance_csv_path, index=False)
    print(f"[OK] Saved per-instance records to {instance_csv_path}")
    
    for n, tours_list in tours_by_size.items():
        tours_array = np.array(tours_list, dtype=object)
        tours_path = os.path.join(DRIVE_RESULTS_DIR, f'concorde_lengths_tsp_{n}.npy')
        np.save(tours_path, tours_array)
        print(f"[OK] Saved tours for TSP-{n} to {tours_path}")
        
    config_data = {
        'TSP_SIZES': TSP_SIZES,
        'NUM_EVAL_INSTANCES': NUM_EVAL_INSTANCES,
        'TIME_LIMIT_SEC': TIME_LIMIT_SEC,
        'DISTANCE_SCALE': DISTANCE_SCALE,
        'SEED': SEED,
    }
    config_path = os.path.join(DRIVE_RESULTS_DIR, 'config.json')
    with open(config_path, 'w') as f:
        json.dump(config_data, f, indent=4)
    print(f"[OK] Saved configuration to {config_path}")

## 11. Optimality Report

In [ ]:
if per_instance_records:
    status_counts = df_instances.groupby(['size', 'status']).size().unstack(fill_value=0)
    print(f"-----------------------------------\n--- Optimality Report ---\n-----------------------------------")
    display(status_counts)

## 12. Visualise a Solved Tour

In [ ]:
if tours_by_size and per_instance_records:
    max_size = df_summary['TSP Size'].max()
    dataset_path = os.path.join(DRIVE_DATA_DIR, f'tsp{max_size}_val.npy')
    
    if os.path.exists(dataset_path):
        dataset = np.load(dataset_path)
        if NUM_EVAL_INSTANCES is not None and len(dataset) > NUM_EVAL_INSTANCES:
            np.random.seed(SEED)
            idx = np.random.choice(len(dataset), NUM_EVAL_INSTANCES, replace=False)
            dataset = dataset[idx]
            
        inst_idx = 0
        coords = dataset[inst_idx]
        tour = tours_by_size[max_size][inst_idx]
        
        if tour:
            tour_coords = coords[tour]
            tour_coords = np.vstack([tour_coords, tour_coords[0]])
            
            plt.figure(figsize=(6, 6))
            plt.plot(tour_coords[:, 0], tour_coords[:, 1], marker='o', color='b', linestyle='-')
            plt.plot(coords[tour[0], 0], coords[tour[0], 1], marker='s', color='r', markersize=10, label='Start/End')
            plt.title(f'Sample Concorde Tour (TSP-{max_size})')
            plt.xlabel('X')
            plt.ylabel('Y')
            plt.legend()
            plt.grid(True, linestyle=':')
            plt.show()
        else:
            print(f"[WARNING] No valid tour for visualization.")

## Output Structure

Files generated in the baseline results directory (`DRIVE_RESULTS_DIR`):
```text
baseline_results/
├── config.json
├── concorde_summary.csv
├── concorde_per_instance.csv
├── concorde_lengths_tsp_10.npy
├── concorde_lengths_tsp_20.npy
└── ...
```

Example loading code for gap computation:
```python
import numpy as np
import pandas as pd

# Load summary
concorde_summary = pd.read_csv('/content/drive/MyDrive/TSP_Data/baseline_results/concorde_summary.csv')

# Load exact tours for TSP-50
concorde_tours_50 = np.load('/content/drive/MyDrive/TSP_Data/baseline_results/concorde_lengths_tsp_50.npy', allow_pickle=True)
```